<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/01_document_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 47.9 MB/s eta 0:00:00


In [6]:
import fitz
import json
import re
from pathlib import Path
from google.colab import files

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
uploaded = files.upload()
pdf_file= [
    filename
    for filename in uploaded
    if filename.lower().endswith(".pdf")
]
if not pdf_file:
  raise ValueError("Please upload atleast one PDF file")
print("Uploaded PDFs: ", pdf_file)

Saving 12396518.pdf to 12396518 (1).pdf
Uploaded PDFs:  ['12396518 (1).pdf']


In [9]:
def extract_pdf(pdf_path):
  """ Extract text from each page of the PAF"""
  pages=[]

  with fitz.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf, start=1):
      text= page.get_text("text").strip()
      if text:
        pages.append(
            {
                "text": text,
                "page": page_number,
                "source" : Path(pdf_path).name

            }
        )
  return pages

# Extracting the first uploaded PDF
pdf_path = pdf_file[0]
pages = extract_pdf(pdf_path)

print("PDF: ",pdf_path)
print("Pages containing text: ", len(pages))
print("First page preview:\n")
print(pages[0]["text"][:200] if pages else "No selectable text found")

PDF:  12396518 (1).pdf
Pages containing text:  12
First page preview:

Published online 23 August 2002
Behavioural models of population growth rates:
implications for conservation and prediction
William J. Sutherland1* and Ken Norris2
1Centre for Ecology, Evolution and C


In [10]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
  """Spliting text into overlapping character chunks"""
  if chunk_size <=0:
    raise ValueError("chunk size must be positive")
  if overlap <0 or overlap >=chunk_size:
    raise ValueError(" overlap must be >=0 and <chunk_size")

  chunks = []
  start = 0

  while start<len(text):
    end = min(start + chunk_size, len(text))

    if end < len(text):
      boundary = text.rfind(" ", start, end)
      if boundary > start:
        end = boundary

    chunk = text[start:end].strip()

    if chunk:
      chunks.append(chunk)

    if end >= len(text):
      break

    start = end - overlap
  return chunks

sample_chunks = chunk_text(pages[0]["text"])

print("Number of chunks:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk[:500])


Number of chunks: 8

--- Chunk 1 ---
Published online 23 August 2002
Behavioural models of population growth rates:
implications for conservation and prediction
William J. Sutherland1* and Ken Norris2
1Centre for Ecology, Evolution and Conservation, University of East Anglia, Norwich NR4 7TJ, UK
2School of Animal & Microbial Sciences, University of Reading, Whiteknights, PO Box 228, Reading, RG6 6AJ, UK
Conservation biologists often wish to predict how vertebrate populations will respond to local or global
changes in conditions suc

--- Chunk 2 ---
or from conservation activities. Predicting the conse-
quences of such changes almost always requires understanding the population growth rate and the density
dependence. Traditional means of directly measuring density dependence are often extremely difﬁcult
and have the problem that if the environment changes then it is necessary to remeasure the density
dependence. We describe an alternative approach that does not require such long dataset

In [13]:
all_chunks = []

for page_data in pages:
  page_chunks = chunk_text(page_data["text"])

  for chunk_index, chunk in enumerate(page_chunks):
    record={
        "id":(
            f"{Path(page_data['source']).stem}"
            f"_p{page_data['page']}"
            f"_c{chunk_index}"
        ),
        "text": chunk,
        "source": page_data["source"],
        "page": page_data["page"],
        "chunk_index": chunk_index

    }
    all_chunks.append(record)

print("Total chunks: ",len(all_chunks))

if all_chunks:
  print("\n Example record: ")
  print(json.dumps(all_chunks[0], indent=2, ensure_ascii=False))

Total chunks:  102

 Example record: 
{
  "id": "12396518 (1)_p1_c0",
  "text": "Published online 23 August 2002\nBehavioural models of population growth rates:\nimplications for conservation and prediction\nWilliam J. Sutherland1* and Ken Norris2\n1Centre for Ecology, Evolution and Conservation, University of East Anglia, Norwich NR4 7TJ, UK\n2School of Animal & Microbial Sciences, University of Reading, Whiteknights, PO Box 228, Reading, RG6 6AJ, UK\nConservation biologists often wish to predict how vertebrate populations will respond to local or global\nchanges in conditions such as those resulting from sea-level rise, deforestation, exploitation, genetically\nmodiﬁed crops, global warming, human disturbance or from conservation activities. Predicting the conse-\nquences of such changes almost always requires understanding the population growth rate and the",
  "source": "12396518 (1).pdf",
  "page": 1,
  "chunk_index": 0
}


In [15]:
from pathlib import Path

output_dir = Path("/content/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "document_chunks.jsonl"

with output_path.open("w", encoding="utf-8") as f:
  for record in all_chunks:
    f.write(
        json.dumps(record, ensure_ascii=False) + "\n"
    )

print(f"Saved {len(all_chunks)} chunks to  {output_path}")

Saved 102 chunks to  /content/processed/document_chunks.jsonl


In [16]:
#verify

loaded_chunks=[]

with output_path.open("r", encoding="utf-8") as f:
  for line in f:
    loaded_chunks.append(json.loads(line))
assert len(loaded_chunks) == len(all_chunks)
assert all(
    {"id", "text","source","page","chunk_index"} <= record.keys()
    for record in loaded_chunks
)
print("Validation Passed!")
print("Records: ", len(loaded_chunks))
print("Uniqur IDs: ", len({r["id"] for r in loaded_chunks}))

Validation Passed!
Records:  102
Uniqur IDs:  102


In [17]:
files.download(str(output_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
#Saving to drive
from google.colab import drive

drive.mount("/content/drive")

drive_output = Path(
    "/content/drive/MyDrive/HybridGraphRAG"
)
drive_output.mkdir(parents=True, exist_ok=True)

drive_output_file = (
    drive_output / "document_chunks.jsonl"
)

import shutil
shutil.copy2(output_path, drive_output_file)

print("Saved to Google Drive:", drive_output_file)

Mounted at /content/drive
Saved to Google Drive: /content/drive/MyDrive/HybridGraphRAG/document_chunks.jsonl
